In [ ]:
import os, sys
from pathlib import Path

# Ensure project root is importable (handles running from /notebooks subfolder)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils").exists() and (PROJECT_ROOT.parent / "utils").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Organization context
ORG_ID = os.environ.get("ORGANIZATION_ID", "").strip()
if not ORG_ID:
    raise ValueError("Set ORGANIZATION_ID env var before running notebooks.")

# Download latest CSVs from Storage into data_exports/{ORG_ID}/
from scripts.sync_storage import sync_org_exports
LOCAL_ORG_EXPORT_DIR = sync_org_exports(ORG_ID)  # returns Path("data_exports/<org_id>")
print("✅ Local org export folder:", LOCAL_ORG_EXPORT_DIR)

# 04_client_best_time.ipynb
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta

from utils.io_utils import should_run, save_state, mark_state_ran, write_output_json, read_csv_to_df
from utils.feature_utils import get_day_name

DATA_DIR = str(LOCAL_ORG_EXPORT_DIR)
STATE_PATH = "state/04_state.json"
OUT_PATH = "outputs/client/best_time_to_visit.json"

input_files = [f"{DATA_DIR}/visits.csv", f"{DATA_DIR}/services.csv"]

run, new_state = should_run(input_files, STATE_PATH)
if not run:
    write_output_json(OUT_PATH, {"updated": False})
    raise SystemExit("No new CSV changes detected.")

# Load (client-safe: only visits + services)
visits = read_csv_to_df(f"{DATA_DIR}/visits.csv", date_cols=["timestamp"])
services = read_csv_to_df(f"{DATA_DIR}/services.csv")

org_id = None
if "organization_id" in services.columns and len(services):
    org_id = services["organization_id"].dropna().iloc[0] if services["organization_id"].notna().any() else None

# Slot baseline model: congestion = 0.6*traffic_norm + 0.4*wait_norm
slot = visits.groupby(["dow","hour"], as_index=False).agg(
    avg_traffic=("visit_id","count"),
    avg_wait=("wait_time_minutes","mean")
)

t = slot["avg_traffic"].astype(float)
w = slot["avg_wait"].fillna(0).astype(float)

t_norm = (t - t.min()) / (t.max() - t.min() + 1e-9)
w_norm = (w - w.min()) / (w.max() - w.min() + 1e-9)

slot["level"] = (0.6*t_norm + 0.4*w_norm).clip(0,1)

# Weekly pattern
weekly_pattern = []
for d in sorted(slot["dow"].unique()):
    s = slot[slot["dow"] == d].sort_values("level")
    best = s.iloc[0]
    worst = s.iloc[-1]
    weekly_pattern.append({
        "dow": int(d),
        "dow_name": get_day_name(int(d)),
        "best_hour": int(best["hour"]),
        "worst_hour": int(worst["hour"]),
        "avg_traffic": float(round(s["avg_traffic"].mean(), 2))
    })

# Best times this month + predicted congestion (date -> dow profile)
today = visits["timestamp"].max().date()
start_date = today.replace(day=1)
end_date = (start_date + timedelta(days=32)).replace(day=1) - timedelta(days=1)

slot_map = slot.set_index(["dow","hour"])[["level","avg_wait"]].to_dict("index")

def best_windows_for_dow(dow: int):
    s = slot[slot["dow"] == dow].sort_values("level").head(2)
    windows = []
    for _, r in s.iterrows():
        h = int(r["hour"])
        windows.append({
            "start": f"{h:02d}:00",
            "end": f"{min(h+1,23):02d}:30",
            "expected_wait": int(round(float(r["avg_wait"]) if not np.isnan(r["avg_wait"]) else 0))
        })
    return windows

best_times_this_month = []
predicted_congestion = []

d = start_date
while d <= end_date:
    # python weekday(): Mon=0..Sun=6 -> your dow: Sun=0..Sat=6
    py = pd.Timestamp(d).weekday()
    your_dow = (py + 1) % 7

    best_times_this_month.append({
        "date": str(d),
        "dow_name": get_day_name(your_dow),
        "best_windows": best_windows_for_dow(your_dow)
    })

    hourly = []
    for h in range(24):
        rec = slot_map.get((your_dow, h))
        level = float(rec["level"]) if rec else 0.0
        hourly.append({"hour": h, "level": round(level, 2)})

    predicted_congestion.append({"date": str(d), "hourly": hourly})
    d += timedelta(days=1)

# Model metrics (baseline fit on wait_time)
slot_wait = slot.set_index(["dow","hour"])["avg_wait"].to_dict()
visits["pred_wait"] = visits.apply(lambda r: slot_wait.get((int(r["dow"]), int(r["hour"])), np.nan), axis=1)

valid = visits.dropna(subset=["wait_time_minutes","pred_wait"]).copy()
if len(valid):
    mae = float(np.mean(np.abs(valid["wait_time_minutes"] - valid["pred_wait"])))
    var = float(valid["wait_time_minutes"].var())
    if var > 0:
        ss_res = float(((valid["wait_time_minutes"] - valid["pred_wait"])**2).sum())
        ss_tot = float(((valid["wait_time_minutes"] - valid["wait_time_minutes"].mean())**2).sum())
        r2 = 1 - ss_res/ss_tot
    else:
        r2 = None
else:
    mae, r2 = None, None

payload = {
  "generated_at": datetime.now(timezone.utc).isoformat().replace("+00:00","Z"),
  "organization_id": org_id,
  "best_times_this_month": best_times_this_month,
  "weekly_pattern": weekly_pattern,
  "predicted_congestion": predicted_congestion,
  "model_info": {
      "mae": None if mae is None else round(mae, 2),
      "r2": None if r2 is None else round(float(r2), 3)
  },
  "recommendation": "Aim for mid-week mornings for the smoothest visit."
}

write_output_json(OUT_PATH, payload)
save_state(STATE_PATH, mark_state_ran(new_state))

print("✅ 04 complete: outputs/client/best_time_to_visit.json written")
